# Understanding Quantization

**Run in [Google Colab](https://colab.research.google.com/) for GPU access.**<br>
(You can try running it on your computer first. If it's too slow, switch to Google Colab for GPU acceleration.)

<br>

---

<br>

Quantization is one of the most common techniques used to make large models fit on smaller hardware and run faster. In this notebook we'll build an intuition for what it is, how it works under the hood, and see it applied to a real model.



<br>

## 1. Why Quantization?

Every parameter (weight) in a neural network is a number, and that number has to be stored in memory using some numeric format. The most common training format is **FP32** (32-bit floating point) — 4 bytes per number.

The problem: modern models have billions of parameters, and FP32 adds up fast.

Let's do the math for a 7-billion-parameter model at different precisions.

In [1]:
def model_memory_gb(num_params_billion, bits_per_param):
    bytes_per_param = bits_per_param / 8
    total_bytes = num_params_billion * 1e9 * bytes_per_param
    return total_bytes / (1024 ** 3)  # convert to GB

num_params = 7  # billion, e.g. a 7B model like Mistral-7B or SQLCoder-7B

precisions = {
    "FP32 (32-bit)": 32,
    "FP16 / BF16 (16-bit)": 16,
    "INT8 (8-bit)": 8,
    "INT4 (4-bit)": 4,
}

print(f"Memory required to load a {num_params}B-parameter model:\n")
for name, bits in precisions.items():
    gb = model_memory_gb(num_params, bits)
    print(f"  {name:<22} ~ {gb:.1f} GB")


Memory required to load a 7B-parameter model:

  FP32 (32-bit)          ~ 26.1 GB
  FP16 / BF16 (16-bit)   ~ 13.0 GB
  INT8 (8-bit)           ~ 6.5 GB
  INT4 (4-bit)           ~ 3.3 GB


A 7B model needs **~28 GB** at FP32 — more than most consumer and even many cloud GPUs have available. At 4-bit, the same model needs **~3.5 GB**.

This is the core motivation for quantization: **shrink the numeric representation of the weights so the model fits in available memory and runs faster**, ideally with minimal loss in output quality.

> **Important distinction:** Quantization is applied to a model's weights. It does not change what the model has learned — it changes how precisely those learned values are *stored and computed*.

<br>

## 2. What Is Quantization, Conceptually?

Floating-point numbers (like FP32) can represent a huge range of values with high precision. Quantization **maps** those high-precision values onto a much smaller set of discrete values — typically low-bit integers.

**Analogy:** 
- Imagine measuring temperature. A precise thermometer might read `21.847°C`. 
- If you round to the nearest whole degree, you get `22°C`. 
- You've lost some precision, but you can still tell hot from cold, and you save a lot of space if you only ever need to store whole numbers.

Quantization does the same thing to model weights: it rounds a wide range of float values into a small number of integer "buckets," while trying to preserve the overall shape/relationships of the original data.


<br>

## 3. Precision Types Walkthrough

| Format | Bits | Typical Use |
|---|---|---|
| FP32 | 32 | Default training precision |
| FP16 / BF16 | 16 | Common for training/inference speedups, minimal quality loss |
| INT8 | 8 | Common for inference, noticeable memory savings |
| INT4 (e.g. NF4) | 4 | Aggressive compression, used to fit very large models on limited hardware |

<br>

![](../_images/quantization-example.webp)

<br>

As we go from FP32 → INT4:
- **Memory footprint shrinks** (roughly linearly with bit-width)
- **Inference can get faster** (less data to move around, sometimes faster integer math)
- **Precision decreases**, which *can* translate into lower-quality outputs — the risk grows as bit-width drops

There's no universally "correct" precision — it's a tradeoff decision based on your hardware constraints and how sensitive your task is to small numerical errors.

<br>

## 4. Post-Training Quantization vs. Quantization-Aware Training

There are two main ways quantization gets applied:

- **Post-Training Quantization (PTQ)**: Take an already-trained model and quantize its weights *after* training is complete, typically right before inference.
- **Quantization-Aware Training (QAT)**: Simulate the effects of quantization *during* training, so the model's weights adapt to the precision loss as it learns. This usually produces better results at very low bit-widths, but requires retraining the model — more time and compute.

**In practice:** most people quantizing an existing open-source model (e.g., loading a model from Hugging Face to run on a single GPU) are doing PTQ. That's the pattern you'll see used with tools like `bitsandbytes`.

<br>

## 5. Hands-On: Quantizing a Real Model

Now let's apply this to an actual model. We'll load a small open-source model twice:
1. Once at full precision
2. Once with 4-bit quantization via `BitsAndBytesConfig`

...and compare memory usage.

> This section requires a GPU runtime (Colab: **Runtime → Change runtime type → GPU**).

In [3]:
!pip install -q -U transformers accelerate bitsandbytes



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# For this demo, we will load "GPT-2", a transformer-based language model developed by OpenAI, known for its text generation capabilities.
# 
# It's a relatively small model, good for demonstrating the mechanics quickly

model_name = "gpt2"


<br>

### 5.1 Load at full precision and check memory footprint

In [8]:
model_fp32 = AutoModelForCausalLM.from_pretrained(model_name)

fp32_params = sum(p.numel() for p in model_fp32.parameters())
fp32_bytes = sum(p.numel() * p.element_size() for p in model_fp32.parameters())

print(f"Parameters: {fp32_params:,}")
print(f"Memory footprint (full precision): {fp32_bytes / (1024**2):.1f} MB")

# del model_fp32
# torch.cuda.empty_cache()


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Parameters: 124,439,808
Memory footprint (full precision): 474.7 MB


<br>

### 5.2 Load with 4-bit quantization and compare

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# bitsandbytes reports memory footprint directly
print(f"Memory footprint (4-bit): {model_4bit.get_memory_footprint() / (1024**2):.1f} MB")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Memory footprint (4-bit): 191.2 MB


<br>

### 5.3 Compare output quality



In [10]:
#
# Full precission model
#

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Quantization is a technique that"
inputs = tokenizer(prompt, return_tensors="pt").to(model_fp32.device)

output = model_fp32.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Quantization is a technique that allows you to create a single-dimensional array of objects.

The first step is to create a new array of objects.

The second


In [11]:
# 
# 4-bit quantization
# 

tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Quantization is a technique that"
inputs = tokenizer(prompt, return_tensors="pt").to(model_4bit.device)

output = model_4bit.generate(**inputs, max_new_tokens=30)
print(tokenizer.decode(output[0], skip_special_tokens=True))


Quantization is a technique that allows you to create a set of data structures that can be used to perform a number of different tasks.

The first thing you need to know


<br>

## 6. Tradeoffs and Limitations

**When quantization helps most:**
- Running large models on limited GPU/CPU memory
- Deploying to edge devices or cost-sensitive inference environments
- Serving many concurrent requests where memory is the bottleneck

**When to be cautious:**
- Very low bit-widths (e.g., 2-bit) can noticeably degrade output quality, especially for smaller models that have less redundancy to spare
- Quantized models may need slightly different prompting or may behave inconsistently compared to full-precision versions
- Quantization is a runtime/deployment concern — it doesn't fix a model that was poorly trained or fine-tuned for the task in the first place

<br>

## 7. Key Takeaways

- Quantization reduces the number of bits used to store model weights, shrinking memory footprint and often speeding up inference
- It works by mapping a wide range of float values onto a smaller set of discrete values, using a scale (and sometimes a zero-point)
- **Post-Training Quantization (PTQ)** is applied to an already-trained model at load time — no retraining needed. This is the approach used with tools like `bitsandbytes`
- **Quantization-Aware Training (QAT)** bakes precision-awareness into training itself, and generally performs better at very low bit-widths, but costs more to produce
- Lower bit-widths mean bigger memory savings but a higher risk of quality degradation — it's a tradeoff to tune based on your hardware and task